# ☀️ **Actividad 1 A2IC — Optimización bajo incertidumbre** — **Solución**
### **EL4114 – Optimización · Departamento de Ingeniería Eléctrica · Universidad de Chile**

# 📝 1) Formulación determinista

En la versión determinista se asume que la demanda y la disponibilidad solar son **conocidas con certeza**, dadas por un único pronóstico. El problema es un MILP que decide simultáneamente el compromiso de unidades y su despacho para satisfacer la demanda al mínimo costo.

**Conjuntos**
- $\mathcal{G} = \{\text{G1, G2, G3}\}$: unidades convencionales
- $\mathcal{T} = \{0,1,\ldots,23\}$: períodos (horas del día)

**Parámetros**
- $P^{\min}_g,\, P^{\max}_g$: límites de potencia [MW]
- $c_g$: costo variable de operación [\$/MWh]
- $c^{nl}_g$: costo de *no-load* (estar encendida sin despacho) [\$/h]
- $c^{su}_g$: costo de arranque [\$]
- $D_t$: demanda en la hora $t$ [MW]
- $P^{sol}_t$: generación solar disponible en la hora $t$ [MW]
- VOLL: costo de energía no suministrada [\$/MWh]

**Variables de compromiso** (binarias):
$$u_{g,t}\in\{0,1\},\quad v_{g,t}\in\{0,1\},\quad w_{g,t}\in\{0,1\}$$

**Variables de despacho** (continuas):
$$p_{g,t}\ge 0,\quad p^{sol}_{t}\ge 0,\quad \ell_{t}\ge 0$$

**Función objetivo:**

$$\min_{u,v,w,p}\; \underbrace{\sum_{g,t}\!\left(c^{nl}_g\,u_{g,t}+c^{su}_g\,v_{g,t}\right)}_{\text{costo de compromiso}}
\;+\; \underbrace{\sum_{t}\!\left(\sum_g c_g\,p_{g,t}+\text{VOLL}\cdot\ell_{t}\right)}_{\text{costo de operación}}$$

sujeto a las siguientes restricciones:

**1. Balance de potencia** $(\forall\,t)$:
$$\sum_g p_{g,t} + p^{sol}_{t} + \ell_{t} = D_{t}$$

**2. Mínimo técnico** $(\forall\,g,t)$:
$$p_{g,t} \ge P^{\min}_g\,u_{g,t}$$

**3. Límite superior** $(\forall\,g,t)$:
$$p_{g,t} \le P^{\max}_g\,u_{g,t}$$

**4. Disponibilidad solar** $(\forall\,t)$:
$$p^{sol}_{t} \le P^{sol}_{t}$$

**5. Lógica de transición** $(\forall\,g,t)$:
$$u_{g,t}-u_{g,t-1}=v_{g,t}-w_{g,t}$$

**6. Exclusividad arranque/parada** $(\forall\,g,t)$:
$$v_{g,t}+w_{g,t}\le 1$$

Esta formulación constituye el **caso base** del problema: representa la situación ideal en que el operado

#📝 2) Formulación con reservas

En la formulación base el balance se satisface de forma exacta para un único pronóstico, sin margen ante desviaciones. Para dotar al sistema de un colchón frente a contingencias (salida de una unidad, error de pronóstico), se exige una **reserva** equivalente a una fracción $\alpha$ de la demanda.

No introduce nuevas variables. Se modifica únicamente la ecuación de balance de potencia, sobredimensionando la demanda por un factor $(1+\alpha)$:

$$\sum_g p_{g,t} + p^{sol}_t + \ell_t = (1+\alpha)\,D_t \qquad \forall\,t$$

El sistema queda obligado a comprometer y despachar capacidad suficiente para cubrir $(1+\alpha)$ veces la demanda nominal. Es directo de implementar, pero la reserva queda **implícita**: se reparte proporcionalmente al despacho y no se distingue entre energía efectivamente consumida y margen de seguridad disponible.

# 📝 3) Formulación estocástica

La naturaleza temporal del UC lleva naturalmente a una **formulación de dos etapas**:

- **Primera etapa** (anticipativa, *here-and-now*): se decide el compromiso de unidades $u_{g,t}$ **antes** de conocer la realización de la incertidumbre. Esta decisión es única para todos los escenarios posibles → **restricción de no-anticipatividad**.
- **Segunda etapa** (adaptativa, *wait-and-see*): una vez revelado el escenario $s$, se optimiza el despacho $p_{g,t,s}$ para mínimo costo operacional.

**Conjuntos**
- $\mathcal{G} = \{\text{G1, G2, G3}\}$: unidades convencionales
- $\mathcal{T} = \{0,1,\ldots,23\}$: períodos (horas del día)
- $\mathcal{S}$: escenarios de incertidumbre (demanda y solar)

**Parámetros**
- $P^{\min}_g,\, P^{\max}_g$: límites de potencia [MW]
- $c_g$: costo variable de operación [\$/MWh]
- $c^{nl}_g$: costo de *no-load* (estar encendida sin despacho) [\$/h]
- $c^{su}_g$: costo de arranque [\$]
- $D_{t,s}$: demanda en hora $t$, escenario $s$ [MW]
- $P^{sol}_{t,s}$: generación solar disponible en $t$, $s$ [MW]
- VOLL: costo de energía no suministrada [\$/MWh]

**Variables de primera etapa** (sin índice de escenario):
$$u_{g,t}\in\{0,1\},\quad v_{g,t}\in\{0,1\},\quad w_{g,t}\in\{0,1\}$$

**Variables de segunda etapa** (con índice de escenario):
$$p_{g,t,s}\ge 0,\quad p^{sol}_{t,s}\ge 0,\quad \ell_{t,s}\ge 0$$

**Modelo estocástico de dos etapas (Recourse Problem):**

$$\min_{u,v,w}\; \underbrace{\sum_{g,t}\!\left(c^{nl}_g\,u_{g,t}+c^{su}_g\,v_{g,t}\right)}_{\text{1ª etapa: compromiso}}
\;+\; \underbrace{\sum_{s\in\mathcal{S}}\pi_s\!\sum_{t}\!\left(\sum_g c_g\,p_{g,t,s}+\text{VOLL}\cdot\ell_{t,s}\right)}_{\text{2ª etapa: despacho esperado}}$$

sujeto a las siguientes restricciones:

**1. Balance de potencia** $(\forall\,t,s)$:
$$\sum_g p_{g,t,s} + p^{sol}_{t,s} + \ell_{t,s} = D_{t,s}$$

**2. Mínimo técnico** $(\forall\,g,t,s)$:
$$p_{g,t,s} \ge P^{\min}_g\,u_{g,t}$$

**3. Límite superior** $(\forall\,g,t,s)$:
$$p_{g,t,s} \le P^{\max}_g\,u_{g,t}$$

**4. Disponibilidad solar** $(\forall\,t,s)$:
$$p^{sol}_{t,s} \le P^{sol}_{t,s}$$

**5. Lógica de transición** $(\forall\,g,t)$:
$$u_{g,t}-u_{g,t-1}=v_{g,t}-w_{g,t}$$

**6. Exclusividad arranque/parada** $(\forall\,g,t)$:
$$v_{g,t}+w_{g,t}\le 1$$

La **no-anticipatividad** es implícita: $u_{g,t}$, $v_{g,t}$, $w_{g,t}$ no llevan índice $s$, por lo que la decisión de compromiso es idéntica para todos los escenarios.

# **Instalación de dependencias**
Ejecute esta celda una sola vez (en Colab tarda unos segundos).

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
#pio.renderers.default = "colab"   # cambie a "notebook" si usa Jupyter local, o "browser"

## **Instalación de Gurobi**
Antes de resolver el problema revise el tutorial de gurobi disponible en [link](https://drive.google.com/file/d/1n3iC3TfnXAqgOkLUMMeV3gVTcnegJAvN/view?usp=sharing)

In [2]:
# Subir la licencia a colab (archivo gurobi.lic)
#from google.colab import files
#uploaded = files.upload()

#!pip install gurobipy
import gurobipy as grb

In [3]:
# Se crea una variable de entorno para luego acceder a la licencia
#import os
#os.environ['GRB_LICENSE_FILE'] = '/content/gurobi.lic'
#print(os.environ['GRB_LICENSE_FILE'])

# Se verifica la versión de Gurobi que se está usando
#print(grb.gurobi.version())

# Sección 1 — Modelo determinista con reservas


## 1.1 Datos del sistema y visualización del pronóstico base

In [4]:
# ---------------- DATOS DEL SISTEMA ----------------
T = list(range(24))                 # periodos 0..23 (horas)
G = ['G1', 'G2', 'G3']              # unidades convencionales

Pmin   = {'G1': 40,  'G2': 20,  'G3': 10}    # MW
Pmax   = {'G1': 150, 'G2': 100, 'G3': 80}    # MW
cvar   = {'G1': 20,  'G2': 40,  'G3': 70}    # $/MWh  (G1 base barata, G3 peaker cara)
cnl    = {'G1': 100, 'G2': 80,  'G3': 50}    # $/h    costo de estar encendida
cstart = {'G1': 500, 'G2': 300, 'G3': 100}   # $      costo de partida
VOLL   = 10000                               # $/MWh  costo de energia no suministrada

# Perfil horario de demanda (normalizado 0-1) y escala a MW
perfil_D = np.array([0.65,0.60,0.58,0.56,0.58,0.62,0.70,0.80,0.88,0.92,0.95,0.97,
                     0.98,0.96,0.93,0.92,0.94,1.00,0.98,0.95,0.90,0.85,0.78,0.70])
D_base = 248 * perfil_D                       # MW

# Perfil solar: campana diurna (cero de noche, maximo ~mediodia)
Psol_base = np.array([max(0.0, 150*np.sin(np.pi*(h-6)/12)) if 6 <= h <= 18 else 0.0
                      for h in T])

print("Capacidad convencional total:", sum(Pmax.values()), "MW")
print("Demanda maxima (base):", round(D_base.max(),1), "MW   |   Solar maxima (base):", round(Psol_base.max(),1), "MW")

Capacidad convencional total: 330 MW
Demanda maxima (base): 248.0 MW   |   Solar maxima (base): 150.0 MW


In [5]:
# Visualizacion del pronostico determinista (base)
fig = go.Figure()
fig.add_bar(x=T, y=Psol_base, name="Solar disponible", marker_color="#f4a261")
fig.add_scatter(x=T, y=D_base, name="Demanda", mode="lines+markers",
                line=dict(color="#264653", width=3))
fig.add_hline(y=sum(Pmax.values()), line_dash="dash", line_color="crimson",
              annotation_text="Capacidad convencional total")
fig.update_layout(title="Sección 1 — Pronóstico determinista (escenario base)",
                  xaxis_title="Hora", yaxis_title="MW", template="plotly_white",
                  legend=dict(orientation="h"))
fig.show()

## 1.2 Desarrolle una función llamada resolver_uc_reserva() en Gurobi que resuelva el problema de Unit Commitment para distintos niveles de reservas (utilice el enfoque A, $\alpha$ será parámetro de entrada de la función). La función debe retornar los valores óptimos de las variables de decisión (estados de encendido, despachos de potencia y energía no suministrada).

In [6]:
def resolver_uc_reserva(D_nom, Sol_nom, reserva=0.15, nombre="UC_Reserva"):
    """
    UC determinista con restriccion de reserva de capacidad.
    D_nom, Sol_nom : arrays de 24 valores (un solo pronostico).
    reserva        : fraccion de reserva (0.15 = 15%).
    """
    m = grb.Model(nombre)
    m.Params.OutputFlag = 0
    u   = m.addVars(G, T, vtype=grb.GRB.BINARY, name="u")
    v   = m.addVars(G, T, vtype=grb.GRB.BINARY, name="v")
    w   = m.addVars(G, T, vtype=grb.GRB.BINARY, name="w")
    p   = m.addVars(G, T, lb=0, name="p")
    ps  = m.addVars(T, lb=0, name="ps")
    ell = m.addVars(T, lb=0, name="ell")

    # Objetivo: compromiso + operacion determinista
    m.setObjective(
        grb.quicksum(cnl[g]*u[g,t] + cstart[g]*v[g,t] for g in G for t in T)
        + grb.quicksum(cvar[g]*p[g,t] for g in G for t in T)
        + grb.quicksum(VOLL*ell[t] for t in T),
        grb.GRB.MINIMIZE)

    for g in G:
        for t in T:
            ut_1 = 0 if t == 0 else u[g, t-1]
            m.addConstr(u[g,t] - ut_1 == v[g,t] - w[g,t])
            m.addConstr(v[g,t] + w[g,t] <= 1)
            m.addConstr(p[g,t] >= Pmin[g]*u[g,t])
            m.addConstr(p[g,t] <= Pmax[g]*u[g,t])
    for t in T:
        m.addConstr(ps[t] <= Sol_nom[t])
        if reserva > 0:
          m.addConstr(grb.quicksum(p[g,t] for g in G) + ps[t] + ell[t] == (1+reserva)*D_nom[t])
        else:
          m.addConstr(grb.quicksum(p[g,t] for g in G) + ps[t] + ell[t] == D_nom[t])

    m.optimize()

    status_map = {grb.GRB.OPTIMAL: "Optimal", grb.GRB.INFEASIBLE: "Infeasible",
                  grb.GRB.UNBOUNDED: "Unbounded"}
    return dict(
        status=status_map.get(m.Status, f"Status_{m.Status}"),
        obj=m.ObjVal,
        u={(g, t): round(u[g,t].X) for g in G for t in T},
        v={(g, t): round(v[g,t].X) for g in G for t in T},
        w={(g, t): round(w[g,t].X) for g in G for t in T},
        p={(g, t): p[g,t].X for g in G for t in T},
        ps={t: ps[t].X for t in T},
        ell={t: ell[t].X for t in T},
    )

## 1.3 Ejecute el modelo de forma iterativa para evaluar cuatro niveles de margen de reserva: $R \in \{0\%,\,10\%,\,15\%,\,20\%\}$. El caso con $0\%$ de reserva actuará como la base determinista pura sin criterios de seguridad.



In [7]:
# Resolver con distintos margenes de reserva (R=0 es el determinista puro)
alphas = [0.0, 0.10, 0.15, 0.20]
resultados_reserva = {}
for alpha in alphas:
    res = resolver_uc_reserva(D_base, Psol_base, reserva=alpha,
                              nombre=f"Reserva_{int(alpha*100)}pct")
    resultados_reserva[alpha] = res
    n_on = sum(res["u"][(g, t)] for g in G for t in T)
    ens_tot = sum(res["ell"][t] for t in T)
    print(f"Reserva {alpha*100:4.0f}%: costo=${res['obj']:>10,.1f}  "
          f"unidades-hora ON={n_on:>3.0f}  ENS={ens_tot:.1f} MWh  [{res['status']}]")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2644077
Academic license 2644077 - for non-commercial use only - registered to di___@ug.uchile.cl
Reserva    0%: costo=$  90,463.9  unidades-hora ON= 36  ENS=0.0 MWh  [Optimal]
Reserva   10%: costo=$ 107,391.3  unidades-hora ON= 43  ENS=0.0 MWh  [Optimal]
Reserva   15%: costo=$ 116,852.1  unidades-hora ON= 46  ENS=0.0 MWh  [Optimal]
Reserva   20%: costo=$ 126,790.0  unidades-hora ON= 49  ENS=0.0 MWh  [Optimal]


## 1.4 Realice las siguientes gráficas:
1) Un diagrama de área apilada para el caso sin reserva (0%) y el caso más conservador (20%) que muestre el aporte de cada unidad térmica, la generación solar utilizada y la Energía No Suministrada (ENS) frente a la curva de demanda.
2) Una comparación visual ( del estado de encendido/apagado de las unidades entre el escenario sin reserva (0%) y el escenario conservador (20%).
3) Un gráfico de barras que compare el costo total de operación del sistema para cada uno de los cuatro márgenes de reserva evaluados.

In [8]:
# Despacho del modelo determinista puro (R=0%)
def plot_despacho_det(res, D_s, titulo):
    """Grafico de area apilada: aporte de cada unidad + solar + ENS vs demanda."""
    fig = go.Figure()
    colores = {"G1": "#2a9d8f", "G2": "#e9c46a", "G3": "#e76f51"}
    for g in G:
        fig.add_scatter(x=T, y=[res["p"][(g, t)] for t in T], name=g,
                        stackgroup="one", mode="none", fillcolor=colores[g])
    fig.add_scatter(x=T, y=[res["ps"][t] for t in T], name="Solar usada",
                    stackgroup="one", mode="none", fillcolor="#f4a261")
    fig.add_scatter(x=T, y=[res["ell"][t] for t in T], name="No suministrada",
                    stackgroup="one", mode="none", fillcolor="#9b2226")
    fig.add_scatter(x=T, y=D_s, name="Demanda", mode="lines+markers",
                    line=dict(color="black", width=2, dash="dot"))
    fig.update_layout(title=titulo, xaxis_title="Hora", yaxis_title="MW",
                      template="plotly_white", legend=dict(orientation="h"))
    fig.show()

plot_despacho_det(resultados_reserva[0.0], D_base,
                  "Sección 1 — Despacho determinista (R=0%, sin reservas)")

plot_despacho_det(resultados_reserva[0.20], D_base,
                  "Sección 1 — Despacho determinista (R=20%, más conservador)")

In [9]:
# Heatmaps de compromiso: reserva 0% vs 20%
fig = make_subplots(rows=1, cols=2, subplot_titles=["R=0% (sin margen)", "R=20% (conservador)"])
for j, alpha in enumerate([0.0, 0.20]):
    res = resultados_reserva[alpha]
    M = np.array([[res["u"][(g, t)] for t in T] for g in G])
    fig.add_trace(go.Heatmap(z=M, x=[f"h{t}" for t in T], y=G,
                             colorscale=[[0, "#e9ecef"], [1, "#e76f51"]],
                             showscale=False, xgap=1, ygap=3), row=1, col=j+1)
fig.update_layout(title="Sección 1 — Compromiso de unidades: sin reserva vs conservador",
                  template="plotly_white", height=280)
fig.show()

# Grafico de barras: costo por nivel de reserva
fig = go.Figure(go.Bar(
    x=[f"{a*100:.0f}%" for a in alphas],
    y=[resultados_reserva[a]["obj"] for a in alphas],
    marker_color=["#94a3b8", "#e9c46a", "#e76f51", "#9b2226"],
    text=[f"${resultados_reserva[a]['obj']:,.0f}" for a in alphas],
    textposition="outside"))
fig.update_layout(title="Sección 1 — Trade-off: costo total vs margen de reserva",
                  xaxis_title="Margen de reserva (R)", yaxis_title="Costo total [$]",
                  template="plotly_white", height=400)
fig.show()

# Sección 2 -  Modelo estocástico mediante SAA

## 1.1 Generación de escenarios
En vez de usar un único pronóstico con reservas conservadoras, modelamos la incertidumbre con **distribuciones de probabilidad** y aplicamos **Sample Average Approximation (SAA)**:
- Factor de demanda: $k_D \sim \mathcal{N}(1.0,\; 0.20^2)$
- Factor solar: $k_S \sim \mathcal{N}(1.0,\; 0.50^2)$ (mayor incertidumbre en energía solar)
- Correlación negativa: $\rho = -0.5$ (día nublado → menos solar, más demanda)

El método SAA consiste en:
1. Muestrear $N$ realizaciones $(k_D^{(i)}, k_S^{(i)})$ desde la distribución conjunta
2. Construir escenarios: $D_t^{(i)} = k_D^{(i)} \cdot D_t^{\text{base}}$, $\;P_t^{sol,(i)} = k_S^{(i)} \cdot P_t^{sol,\text{base}}$
3. Resolver el UC estocástico con estos $N$ escenarios equiprobables ($\pi_i = 1/N$)


In [10]:
def generar_escenarios_SAA(N, seed=42):
    """Muestrea N escenarios de (kD, kS) desde distribucion normal bivariada."""
    rng = np.random.default_rng(seed)
    mu = [1.0, 1.0]
    sigma_D, sigma_S, rho = 0.1, 0.3, -0.5
    cov = [[sigma_D**2, rho*sigma_D*sigma_S],
           [rho*sigma_D*sigma_S, sigma_S**2]]
    samples = rng.multivariate_normal(mu, cov, size=N)
    samples[:, 0] = np.clip(samples[:, 0], 0.5, 1.6)   # limites fisicos demanda
    samples[:, 1] = np.clip(samples[:, 1], 0.0, 2.0)   # limites fisicos solar

    D_saa, Sol_saa, pi_saa = {}, {}, {}
    for i in range(N):
        s = f"saa_{i}"
        D_saa[s] = samples[i, 0] * D_base
        Sol_saa[s] = samples[i, 1] * Psol_base
        pi_saa[s] = 1.0 / N
    return D_saa, Sol_saa, pi_saa, samples

In [11]:
# Visualizacion de escenarios muestreados (N=100)
_, _, _, muestras_100 = generar_escenarios_SAA(100, seed=42)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Muestras en espacio (kD, kS)",
                                    "Perfiles de demanda muestreados"])

# Scatter de factores
fig.add_trace(go.Scatter(x=muestras_100[:, 0], y=muestras_100[:, 1], mode="markers",
                         marker=dict(size=6, color="#2a9d8f", opacity=0.7),
                         name="Muestras"), row=1, col=1)
fig.add_trace(go.Scatter(x=[1.0], y=[1.0], mode="markers",
                         marker=dict(size=12, color="red", symbol="x"),
                         name="Base (1,1)"), row=1, col=1)

# Spaghetti de perfiles de demanda
D_100, _, _, _ = generar_escenarios_SAA(100, seed=42)
for i, s in enumerate(D_100):
    fig.add_trace(go.Scatter(x=T, y=D_100[s], mode="lines",
                             line=dict(color="#264653", width=0.5), opacity=0.2,
                             showlegend=(i == 0), name="Escenarios",
                             legendgroup="esc"), row=1, col=2)
fig.add_trace(go.Scatter(x=T, y=D_base, mode="lines",
                         line=dict(color="red", width=2.5, dash="dash"),
                         name="Base"), row=1, col=2)

fig.update_xaxes(title_text="kD (factor demanda)", row=1, col=1)
fig.update_yaxes(title_text="kS (factor solar)", row=1, col=1)
fig.update_xaxes(title_text="Hora", row=1, col=2)
fig.update_yaxes(title_text="MW", row=1, col=2)
fig.update_layout(title="Sección 2 — Escenarios muestreados desde distribución bivariada (N=100)",
                  template="plotly_white", height=420, legend=dict(orientation="h"))
fig.show()

## 1.2 Implemente una función llamada resolver_uc() que resuelva el problema de Unit Commitment estocástico de dos etapas. El código debe estructurarse de modo que las decisiones de compromiso (encendido, apagado y arranque de unidades) se consideren variables de primera etapa (únicas para todo el problema), mientras que los niveles de despacho por unidad, la energía solar aprovechada y la energía no suministrada actúen como variables de segunda etapa (específicas para cada escenario generado).

In [12]:
def resolver_uc(D_in, Sol_in, pi_in, fijar_uc=None, nombre="UC"):
    """
    Resuelve el Unit Commitment (de dos etapas si hay >1 escenario).
    D_in, Sol_in : dict {escenario: array de 24 valores}
    pi_in        : dict {escenario: probabilidad}
    fijar_uc     : dict {(g,t): 0/1} para FIJAR el compromiso (usado en evaluación OOS). None = libre.
    """
    Sset = list(D_in.keys())
    m = grb.Model(nombre)
    m.Params.OutputFlag = 0

    # --- Variables de 1a etapa (NO dependen del escenario) ---
    u = m.addVars(G, T, vtype=grb.GRB.BINARY, name="u")
    v = m.addVars(G, T, vtype=grb.GRB.BINARY, name="v")
    w = m.addVars(G, T, vtype=grb.GRB.BINARY, name="w")
    # --- Variables de 2a etapa (SI dependen del escenario s) ---
    p   = m.addVars(G, T, Sset, lb=0, name="p")
    ps  = m.addVars(T, Sset, lb=0, name="ps")
    ell = m.addVars(T, Sset, lb=0, name="ell")

    # --- Funcion objetivo ---
    costo_compromiso = grb.quicksum(cnl[g]*u[g,t] + cstart[g]*v[g,t] for g in G for t in T)
    costo_operacion  = grb.quicksum(
        pi_in[s]*(grb.quicksum(cvar[g]*p[g,t,s] for g in G for t in T)
                  + grb.quicksum(VOLL*ell[t,s] for t in T))
        for s in Sset)
    m.setObjective(costo_compromiso + costo_operacion, grb.GRB.MINIMIZE)

    # --- Restricciones ---
    for g in G:
        for t in T:
            ut_1 = 0 if t == 0 else u[g, t-1]         # estado inicial: apagada
            m.addConstr(u[g,t] - ut_1 == v[g,t] - w[g,t])  # logica encendido/apagado
            m.addConstr(v[g,t] + w[g,t] <= 1)
            for s in Sset:
                m.addConstr(p[g,t,s] >= Pmin[g]*u[g,t])    # minimo tecnico si esta ON
                m.addConstr(p[g,t,s] <= Pmax[g]*u[g,t])    # maximo / apagada => 0
    for t in T:
        for s in Sset:
            m.addConstr(ps[t,s] <= Sol_in[s][t])           # solo se usa solar disponible
            m.addConstr(grb.quicksum(p[g,t,s] for g in G)  # BALANCE de potencia
                        + ps[t,s] + ell[t,s] == D_in[s][t])

    if fijar_uc is not None:                           # fijar compromiso (para evaluación OOS)
        for g in G:
            for t in T:
                m.addConstr(u[g,t] == fijar_uc[(g, t)])

    m.optimize()

    status_map = {grb.GRB.OPTIMAL: "Optimal", grb.GRB.INFEASIBLE: "Infeasible",
                  grb.GRB.UNBOUNDED: "Unbounded"}
    return dict(
        status=status_map.get(m.Status, f"Status_{m.Status}"),
        obj=m.ObjVal,
        u={(g, t): round(u[g,t].X)   for g in G for t in T},
        v={(g, t): round(v[g,t].X)   for g in G for t in T},
        w={(g, t): round(w[g,t].X)   for g in G for t in T},
        p={(g, t, s): p[g,t,s].X     for g in G for t in T for s in Sset},
        ps={(t, s): ps[t,s].X        for t in T for s in Sset},
        ell={(t, s): ell[t,s].X      for t in T for s in Sset},
        Sset=Sset)

## 1.3 Ejecutar el algoritmo iterativamente evaluando los tamaños de muestra: 2, 5, 10, 25, 50 y 100 escenarios equiprobables. Para cada caso, reporten el costo de la función objetivo obtenido dentro de la muestra.

In [13]:
# Resolver SAA con distintos N
Ns = [2, 5, 10, 25, 50, 100]
resultados_saa = {}

print("Resolviendo SAA para distintos N...")
for N in Ns:
    D_n, Sol_n, pi_n, muestras = generar_escenarios_SAA(N, seed=42)
    saa = resolver_uc(D_n, Sol_n, pi_n, nombre=f"SAA_{N}")
    resultados_saa[N] = {"res": saa, "muestras": muestras}
    n_on = sum(saa["u"][(g, t)] for g in G for t in T)
    print(f"  SAA N={N:>3}: costo in-sample=${saa['obj']:>10,.1f}  "
          f"unidades-hora ON={n_on:>3.0f}  [{saa['status']}]")

Resolviendo SAA para distintos N...
  SAA N=  2: costo in-sample=$  83,239.7  unidades-hora ON= 37  [Optimal]
  SAA N=  5: costo in-sample=$  92,564.7  unidades-hora ON= 46  [Optimal]
  SAA N= 10: costo in-sample=$  91,916.3  unidades-hora ON= 50  [Optimal]
  SAA N= 25: costo in-sample=$  93,070.6  unidades-hora ON= 50  [Optimal]
  SAA N= 50: costo in-sample=$  97,794.1  unidades-hora ON= 51  [Optimal]
  SAA N=100: costo in-sample=$  98,865.5  unidades-hora ON= 53  [Optimal]


## 1.4 Contraste el esquema de encendido de las unidades térmicas obtenido con una muestra pequeña ($N = 2$) frente al obtenido con la muestra más grande ($N = 100$).

In [14]:
# Comparacion visual del compromiso: SAA N=2 vs SAA N=100
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["SAA N=2 (pocos muestreos)",
                                    "SAA N=100 (muchos muestreos)"])
for j, N in enumerate([2, 100]):
    res = resultados_saa[N]["res"]
    M = np.array([[res["u"][(g, t)] for t in T] for g in G])
    fig.add_trace(go.Heatmap(z=M, x=[f"h{t}" for t in T], y=G,
                             colorscale=[[0, "#e9ecef"], [1, "#264653"]],
                             showscale=False, xgap=1, ygap=3), row=1, col=j+1)
fig.update_layout(title="Sección 2 — Compromiso de unidades: pocos vs muchos muestreos",
                  template="plotly_white", height=280)

# Sección 3 - Oráculo y comparación
El **oráculo** (o "dios") conoce el futuro con certeza total. Para cada escenario posible,
resuelve el UC sabiendo exactamente la demanda y solar → obtiene el **costo mínimo posible**
(Wait-and-See, WS).

**Metodología de evaluación out-of-sample (OOS):**
1. Generar $N_{eval} = 200$ escenarios "verdaderos" desde la distribución
2. **Oráculo**: resolver un UC individual por escenario → cota inferior inalcanzable
3. **Cada modelo**: fijar su compromiso $u_{g,t}$ y re-optimizar el despacho sobre los $N_{eval}$ escenarios
4. **Comparar**: $\text{gap}(\%) = \frac{\text{costo modelo} - \text{costo oráculo}}{\text{costo oráculo}} \times 100$

Los modelos evaluados son:
- Reservas ($R = 0\%, 10\%, 15\%, 20\%$)
- SAA con $N \in \{2, 5, 10, 25, 50, 100\}$

In [15]:
# Generar escenarios de evaluacion out-of-sample
N_eval = 200
D_eval, Sol_eval, pi_eval, _ = generar_escenarios_SAA(N_eval, seed=9999)

# Oraculo: resolver UC por escenario individual (Wait-and-See)
print(f"Calculando oraculo para {N_eval} escenarios...")
ws_costos = {}
for i, s in enumerate(D_eval):
    res_s = resolver_uc({s: D_eval[s]}, {s: Sol_eval[s]}, {s: 1.0}, nombre=f"WS_{s}")
    ws_costos[s] = res_s["obj"]
    if (i+1) % 50 == 0:
        print(f"  ...{i+1}/{N_eval} escenarios resueltos")

WS = sum(pi_eval[s] * ws_costos[s] for s in D_eval)
print(f"\nCosto oraculo (WS) = ${WS:,.1f}")

Calculando oraculo para 200 escenarios...
  ...50/200 escenarios resueltos
  ...100/200 escenarios resueltos
  ...150/200 escenarios resueltos
  ...200/200 escenarios resueltos

Costo oraculo (WS) = $94,402.3


In [16]:
# Evaluar TODOS los modelos fijando su compromiso sobre escenarios OOS
modelos_a_evaluar = {}

# Modelos de reserva
for alpha in alphas:
    nombre = f"Reserva {alpha*100:.0f}%"
    modelos_a_evaluar[nombre] = resultados_reserva[alpha]["u"]

# SAA con distintos N
for N in Ns:
    modelos_a_evaluar[f"SAA N={N}"] = resultados_saa[N]["res"]["u"]

# Evaluar cada modelo
evaluacion = {}
print("Evaluando modelos en escenarios OOS...")
for nombre, uc_fijo in modelos_a_evaluar.items():
    oos = resolver_uc(D_eval, Sol_eval, pi_eval, fijar_uc=uc_fijo, nombre="OOS")
    ens_total = sum(oos["ell"][(t, s)] for t in T for s in oos["Sset"]) / N_eval
    evaluacion[nombre] = {
        "costo_oos": oos["obj"],
        "ens_esperada": ens_total,
        "gap_pct": (oos["obj"] - WS) / WS * 100
    }
    print(f"  {nombre:>20s}: costo=${oos['obj']:>10,.1f}  "
          f"ENS={ens_total:>6.1f} MWh  gap={evaluacion[nombre]['gap_pct']:.2f}%")

print(f"\n  {'Oraculo (WS)':>20s}: costo=${WS:>10,.1f}  "
      f"ENS=  0.0 MWh  gap=0.00%")

Evaluando modelos en escenarios OOS...
            Reserva 0%: costo=$1,022,972.0  ENS=  93.2 MWh  gap=983.63%
           Reserva 10%: costo=$ 617,693.5  ENS=  52.3 MWh  gap=554.32%
           Reserva 15%: costo=$ 423,902.5  ENS=  32.7 MWh  gap=349.04%
           Reserva 20%: costo=$ 336,069.6  ENS=  23.8 MWh  gap=256.00%
               SAA N=2: costo=$1,110,555.8  ENS= 102.0 MWh  gap=1076.41%
               SAA N=5: costo=$ 273,626.0  ENS=  17.6 MWh  gap=189.85%
              SAA N=10: costo=$ 137,810.2  ENS=   3.8 MWh  gap=45.98%
              SAA N=25: costo=$ 137,810.2  ENS=   3.8 MWh  gap=45.98%
              SAA N=50: costo=$ 122,435.0  ENS=   2.2 MWh  gap=29.69%
             SAA N=100: costo=$ 103,272.9  ENS=   0.2 MWh  gap=9.40%

          Oraculo (WS): costo=$  94,402.3  ENS=  0.0 MWh  gap=0.00%


In [17]:
# Grafico de convergencia SAA
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Costo OOS vs N muestras SAA",
                                    "ENS esperada vs N muestras SAA"])

costos_saa = [evaluacion[f"SAA N={N}"]["costo_oos"] for N in Ns]
ens_saa = [evaluacion[f"SAA N={N}"]["ens_esperada"] for N in Ns]

# Costo OOS
fig.add_trace(go.Scatter(x=Ns, y=costos_saa, mode="lines+markers", name="SAA",
                         marker=dict(size=10, color="#264653"),
                         line=dict(width=2.5)), row=1, col=1)
fig.add_hline(y=WS, line_dash="dash", line_color="green", row=1, col=1,
              annotation_text="Oráculo (WS)", annotation_position="bottom right")

# ENS
fig.add_trace(go.Scatter(x=Ns, y=ens_saa, mode="lines+markers", name="SAA (ENS)",
                         marker=dict(size=10, color="#e76f51"),
                         line=dict(width=2.5), showlegend=False), row=1, col=2)

fig.update_xaxes(title_text="N muestras SAA", type="log", row=1, col=1)
fig.update_xaxes(title_text="N muestras SAA", type="log", row=1, col=2)
fig.update_yaxes(title_text="Costo OOS [$]", row=1, col=1)
fig.update_yaxes(title_text="ENS esperada [MWh]", row=1, col=2)
fig.update_layout(title="Sección 3 — Convergencia SAA: pocos muestreos → solución inadecuada",
                  template="plotly_white", height=450, legend=dict(orientation="h"))

In [18]:
# Tabla resumen de todos los modelos
print("=" * 80)
print(f"{'Modelo':>25s} {'Costo OOS ($)':>14s} {'Gap vs Oraculo':>15s}")
print("=" * 80)
print(f"{'Oraculo (WS)':>25s} {WS:>14,.1f} {'0.00%':>15s}")
print("-" * 80)
for nombre, vals in sorted(evaluacion.items(), key=lambda x: x[1]["costo_oos"]):
    print(f"{nombre:>25s} {vals['costo_oos']:>14,.1f} "
          f"{vals['gap_pct']:>14.2f}%")
print("=" * 80)

# Grafico de barras horizontal: todos los modelos ordenados
nombres_ord = sorted(evaluacion.keys(), key=lambda x: evaluacion[x]["costo_oos"])
nombres_plot = ["Oráculo (WS)"] + nombres_ord
costos_plot = [WS] + [evaluacion[n]["costo_oos"] for n in nombres_ord]

colores_plot = ["#2ecc71"]   # oracle = green
for n in nombres_ord:
    if "Reserva" in n:
        colores_plot.append("#e76f51")
    elif "SAA" in n:
        colores_plot.append("#264653")

fig = go.Figure(go.Bar(
    y=nombres_plot, x=costos_plot, orientation="h",
    marker_color=colores_plot,
    text=[f"${c:,.0f}" for c in costos_plot],
    textposition="outside"))
fig.add_vline(x=WS, line_dash="dash", line_color="green", line_width=2,
              annotation_text="Oráculo", annotation_position="top right")
fig.update_layout(title="Sección 3 — Comparación de todos los modelos (evaluación OOS)",
                  xaxis_title="Costo esperado OOS [$$]",
                  template="plotly_white", height=550,
                  margin=dict(l=200))
fig.show()

                   Modelo  Costo OOS ($)  Gap vs Oraculo
             Oraculo (WS)       94,402.3           0.00%
--------------------------------------------------------------------------------
                SAA N=100      103,272.9           9.40%
                 SAA N=50      122,435.0          29.69%
                 SAA N=10      137,810.2          45.98%
                 SAA N=25      137,810.2          45.98%
                  SAA N=5      273,626.0         189.85%
              Reserva 20%      336,069.6         256.00%
              Reserva 15%      423,902.5         349.04%
              Reserva 10%      617,693.5         554.32%
               Reserva 0%    1,022,972.0         983.63%
                  SAA N=2    1,110,555.8        1076.41%
